# Aspire : garde-fous du code d'agent — l'analyseur Roslyn vit DANS la compilation

Nos notebooks précédents (01-05) ont construit une pile d'agent .NET complète : orchestration Aspire, stack GenAI réelle, observabilité, agent streaming, tests d'intégration. Ce notebook attaque la question qui vient après : **quand un agent (LLM) génère du code C# pour cette pile, qu'est-ce qui l'empêche d'écrire du code dangereux ?**

La thèse tient en une ligne de parité :

| Python | C#/.NET |
|---|---|
| `mypy`, `ruff` — garde-fous **HORS** compilation (outil séparé à adopter) | **analyseurs Roslyn** — garde-fous **DANS** la compilation (`dotnet build` lui-même rend le diagnostic) |

En Python, un garde-fou vit dans un lint distinct : il faut installer `ruff`, le configurer, l'ajouter au pipeline. En .NET, l'analyseur se référence comme n'importe quelle dépendance — et le **même `dotnet build` qui produit le binaire** rend le verdict, pour tout le monde, sur la machine de chaque développeur et du premier coup.

**Plan** : nous construisons un vrai analyseur (`DiagnosticAnalyzer` Roslyn, ~40 lignes) qui attrape le pattern de deadlock le plus typique du code d'agent généré — bloquer une `Task` avec `.Result`/`.Wait()` — puis nous le voyons tirer sur deux canaux : le canal *build* (`dotnet build` rend l'avertissement, c'est la thèse) et le canal *API* (un `Verifier` qui compile un source via Roslyn et rend son verdict, comme le fait l'IDE). Nous terminons par le contraste Python et trois exercices.

## Contexte : le deadlock `.Result`/`.Wait()`, blessure classique du code d'agent

Un LLM qui génère du C# « dans le style synchrone » écrit très naturellement :

```csharp
var reponse = AppelerLeLlmAsync(prompt).Result;   // bloque la Task
```

Pourquoi c'est mortel et pas juste inélégant : `.Result` **bloque le thread courant** jusqu'à ce que la tâche finisse. Si cette tâche a besoin, pour finir, d'un thread du même pool — continuation en attente, contexte de synchronisation, handler ASP.NET qui doit libérer son thread — le programme **attend un thread qui ne viendra jamais** : deadlock. Symptôme en production : le service se fige silencieusement, aucun crash, aucun log — juste zéro réponse.

C'est précisément le genre de défaut qu'un garde-fou automatique doit attraper : *localement visible* (une expression), *globalement désastreux* (un service mort). Et c'est un pattern que les modèles de langage produisent de façon récurrente, parce que leurs données d'entraînement débordent de code synchrone. D'où la règle AGENTGUARD001 : **toute Task bloquée synchrone en code d'agent est signalée au build**.

In [1]:
using System;
using System.Diagnostics;
using System.IO;

var here = Directory.GetCurrentDirectory();   // le dossier de la serie Aspire

public static class Shell
{
    // Execute un executable et capture stdout+stderr. Pas de shell
    // intermediaire : cmd.exe reecrit toute ligne portant plus d'une paire
    // de guillemets (lecon du notebook 01) -- on lance donc la commande
    // directement, resolue via le PATH.
    public static string Run(string workDir, string cmd, string args)
    {
        var psi = new ProcessStartInfo
        {
            FileName = cmd,
            Arguments = args,
            WorkingDirectory = workDir,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            UseShellExecute = false,
            CreateNoWindow = true
        };
        using var p = Process.Start(psi)!;
        var stdout = p.StandardOutput.ReadToEnd();
        var stderr = p.StandardError.ReadToEnd();
        if (!p.WaitForExit(180_000)) p.Kill();
        return stdout + stderr;
    }
}
Console.WriteLine($"Repertoire de travail : {Path.GetFileName(here)}");

Repertoire de travail : Aspire


## A1 — Le terrain fautif : un worker d'agent qui bloque sa tâche

Le fichier ci-dessous (`AgentGuard.Demo/Program.cs`, committé avec ce notebook) est une **copie de démonstration** du motif canonique de la série (le worker `Channel` du notebook 04) — avec le défaut **injecté** : là où la série attend la tâche avec `await`, ce code la bloque avec `.Result`, deux fois, « pour simplifier ». C'est le genre de code qu'un agent génère quand on lui demande une version synchrone d'un pipeline asynchrone.

Le fichier ne fait pas planter le notebook : l'exécution du programme elle-même termine (blocage borné sur ce scénario minimal) — le défaut est un **risque structurel**, pas une exception immédiate. C'est exactement pourquoi il faut un analyseur : aucun test d'exécution ne le révélera tant que le deadlock ne s'est pas produit.

In [2]:
var terrain = File.ReadAllText(Path.Combine(here, "AgentGuard.Demo", "Program.cs"));
Console.WriteLine(terrain);

// Terrain fautif : copie de demo du motif StreamingAgent.App (notebook 04).
// Un worker d'agent consomme une Channel -- motif canonique de la serie --
// mais ici le code genere "dans le style synchrone" BLOQUE la tache d'appel
// au LLM avec .Result au lieu de l'attendre. C'est le defaut que l'analyseur
// AGENTGUARD001 doit attraper au `dotnet build` (ce fichier ne leve PAS
// d'exception : le blocage ne deadlock ici qu'un contexte reduit -- le
// notebook explique pourquoi le pattern reste mortel en production).

using System;
using System.Threading.Channels;
using System.Threading.Tasks;

var channel = Channel.CreateUnbounded<string>();
_ = Producer.ProduceAsync(channel.Writer, "traduis cette phrase en anglais");

Console.WriteLine(AgentWorker.TranslateSync(channel.Reader));

public static class AgentWorker
{
    // Genere par agent "pour simplifier" : la tache async est bloquee.
    // Deux blocages .Result -- deux diagnostics attendus au build.
    public static string Translat

### Lecture du terrain

Le worker `TranslateSync` porte les deux blocages, marqués `// AGENTGUARD001` :

- `inbound.ReadAsync().AsTask().Result` — on bloque même la **lecture du canal** d'entrée ;
- `CallLlmAsync(prompt).Result` — on bloque **l'appel au LLM**, la partie la plus lente du pipeline.

Deux expressions, deux endroits où un thread s'endort en tenant le pipeline. La méthode `CallLlmAsync` est correcte (elle est `async`) ; c'est le **consommateur** qui dénature le contrat. Un humain relit rarement ça assez attentivement — l'analyseur, lui, ne rate jamais.

## A2 — L'analyseur : quarante lignes qui vivent dans la compilation

Voici l'intégralité de l'analyseur (`AgentGuard.Analyzers/TaskResultBlockAnalyzer.cs`, committé) — un `DiagnosticAnalyzer` Roslyn réel, pas une simplification :

Il s'abonne aux **expressions d'accès de membre** (`.QuelqueChose`), puis filtre en deux étages : un étage **syntaxique** bon marché (le membre s'appelle-t-il `Result` ou `Wait` ?) et un étage **sémantique** (ce membre appartient-il vraiment au type `Task`/`Task<T>` ?). C'est ce second étage qui fait la différence entre un `grep` et un analyseur : il interroge le **modèle sémantique** — la compréhension des types que le compilateur construit.

In [3]:
var analyzerSrc = File.ReadAllText(Path.Combine(here, "AgentGuard.Analyzers", "TaskResultBlockAnalyzer.cs"));
Console.WriteLine(analyzerSrc);

using System.Collections.Immutable;
using Microsoft.CodeAnalysis;
using Microsoft.CodeAnalysis.CSharp;
using Microsoft.CodeAnalysis.Diagnostics;

namespace AgentGuard.Analyzers;

/// <summary>
/// AGENTGUARD001 : blocage synchrone d'une Task (.Result / .Wait()).
///
/// Pattern typique du code d'agent genere : appeler une tache asynchrone
/// (appel LLM, streaming, canal) depuis du code synchrone en la bloquant.
/// Le garde-fou vit DANS la compilation -- `dotnet build` rend le diagnostic
/// sans aucun outil supplementaire (la these du notebook 06 de la serie Aspire,
/// Epic #10473 axe Roslyn).
/// </summary>
[DiagnosticAnalyzer(LanguageNames.CSharp)]
public sealed class TaskResultBlockAnalyzer : DiagnosticAnalyzer
{
    public const string DiagnosticId = "AGENTGUARD001";

    private static readonly DiagnosticDescriptor Rule = new(
        DiagnosticId,
        "Blocage synchrone d'une Task",
        "Task bloquee de maniere synchrone ({0}) : deadlock potentiel en code d'agent",
   

### Anatomie pas à pas

Trois morceaux portent tout :

1. **`DiagnosticDescriptor Rule`** — la carte d'identité du diagnostic : identifiant (`AGENTGUARD001`), titre, message (avec `{0}` rempli au rapport), catégorie, sévérité. `isEnabledByDefault: true` : le diagnostic est actif dès la référence, sans configuration.
2. **`Initialize` + `RegisterSyntaxNodeAction`** — l'analyseur ne parcourt PAS tout le code : il s'abonne à un seul type de nœud syntaxique (`SimpleMemberAccessExpression`), et Roslyn ne l'appelle que sur ces nœuds. C'est ce qui rend l'analyse quasi gratuite.
3. **`AnalyzeNode`** — les deux étages de filtrage : syntaxique d'abord (`Result` ou `Wait` — un test de chaîne), sémantique ensuite (`GetSymbolInfo` → le membre appartient-il à `Task`/`Task<T>` ?). Le `MetadataName` distingue le vrai `Task<T>` (`` `Task`1` `` en métadonnées) de tout homonyme. Seul un nœud ayant passé les deux étages est rapporté — d'où zéro faux positif sur un type custom qui aurait une propriété `Result`.

La sévérité `Warning` (et non `Error`) est un choix : le build ne casse pas, mais le diagnostic est visible à chaque compilation. Passer en `Error` (via `.editorconfig`) ferait du garde-fou un bloqueur — une décision d'équipe, pas de l'analyseur.

### Exercice 1 — Attraper la variante furtive `GetAwaiter().GetResult()`

Le développeur (ou l'agent) qui découvre l'avertissement sur `.Result` connaît souvent l'échappatoire : `task.GetAwaiter().GetResult()` bloque **exactement autant**, mais l'analyseur actuel ne la voit pas — il ne surveille que les accès directs `.Result`/`.Wait`.

**Objectif** : étendre l'analyseur pour signaler cette variante.

**Indices** :
- `# Indice` : le nœud à surveiller devient `SyntaxKind.InvocationExpression` — `GetResult()` est une *invocation*, pas un simple accès.
- `# Indice` : dans l'invocation, vérifier que le membre invoqué s'appelle `GetResult` **et** que son récepteur est lui-même l'invocation de `GetAwaiter` sur... un `Task` (étage sémantique inchangé).
- `# Etape 1` : écrire un source de test contenant `t.GetAwaiter().GetResult()` et vérifier à la main que le Verifier (section suivante) le rend PROPRE aujourd'hui.
- `# Etape 2` : ajouter l'abonnement et le filtre dans une copie de `TaskResultBlockAnalyzer`.
- `# Etape 3` : recharger l'analyseur dans le projet Demo (`dotnet build`) et vérifier que la variante furtive rend AGENTGUARD001.

In [4]:
// Exercice 1 a completer -- squelette de l'extension a ecrire dans une copie
// de TaskResultBlockAnalyzer.cs. Le TODO marque le point d'insertion.
//
// public override void Initialize(AnalysisContext context)
// {
//     context.RegisterSyntaxNodeAction(AnalyzeInvocation,
//         SyntaxKind.InvocationExpression);          // TODO etudiant
// }
//
// private static void AnalyzeInvocation(SyntaxNodeAnalysisContext ctx)
// {
//     var inv = (InvocationExpressionSyntax)ctx.Node;
//     // TODO etudiant : membre == "GetResult" ?
//     // TODO etudiant : recepteur == invocation de "GetAwaiter" sur un Task ?
//     // TODO etudiant : ctx.ReportDiagnostic(...)
// }
Console.WriteLine("Exercice a completer : etendre l'analyseur a GetAwaiter().GetResult()");

Exercice a completer : etendre l'analyseur a GetAwaiter().GetResult()


## A3 — La thèse : `dotnet build` rend le diagnostic, sans aucun outil supplémentaire

Le projet `AgentGuard.Demo` référence l'analyseur **comme dépendance de diagnostic** — deux attributs dans le `.csproj` :

```xml
<ProjectReference Include="../AgentGuard.Analyzers/AgentGuard.Analyzers.csproj"
                  OutputItemType="Analyzer" ReferenceOutputAssembly="false" />
```

`OutputItemType="Analyzer"` dit à MSBuild : « charge cette DLL comme analyseur Roslyn de CE projet ». `ReferenceOutputAssembly="false"` ajoute : « mais elle n'est pas une dépendance runtime ». C'est tout. Pas de paquet à installer, pas de ligne de commande à retenir, aucune action consciente à faire — le prochain `dotnet build` rend le verdict. C'est la thèse en action.

In [5]:
// -t:Rebuild : force la recompilation complete -- un build incrementale
// (projet deja compile par un passage precedent) ne re-emet PAS les
// diagnostics, et le verdict doit apparaitre a CHAQUE execution.
var build = Shell.Run(here, "dotnet", "build AgentGuard.Demo -t:Rebuild -v q --nologo");
Console.WriteLine(build);

D:\dev\CoursIA-roslyn\MyIA.AI.Notebooks\GenAI\Aspire\AgentGuard.Demo\Program.cs(24,22): warning AGENTGUARD001: Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent [D:\dev\CoursIA-roslyn\MyIA.AI.Notebooks\GenAI\Aspire\AgentGuard.Demo\AgentGuard.Demo.csproj]
D:\dev\CoursIA-roslyn\MyIA.AI.Notebooks\GenAI\Aspire\AgentGuard.Demo\Program.cs(25,16): warning AGENTGUARD001: Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent [D:\dev\CoursIA-roslyn\MyIA.AI.Notebooks\GenAI\Aspire\AgentGuard.Demo\AgentGuard.Demo.csproj]

La génération a réussi.

D:\dev\CoursIA-roslyn\MyIA.AI.Notebooks\GenAI\Aspire\AgentGuard.Demo\Program.cs(24,22): warning AGENTGUARD001: Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent [D:\dev\CoursIA-roslyn\MyIA.AI.Notebooks\GenAI\Aspire\AgentGuard.Demo\AgentGuard.Demo.csproj]
D:\dev\CoursIA-roslyn\MyIA.AI.Notebooks\GenAI\Aspire\AgentGuard.Demo\Program.cs(25,16): warning AGENTGUARD001: Task 

### Lecture du verdict

La sortie du build porte **deux avertissements `AGENTGUARD001`**, un par blocage :

- `Program.cs(24,22)` — le `.Result` sur la lecture du canal ;
- `Program.cs(25,16)` — le `.Result` sur l'appel LLM.

Chacun nomme le fichier, la ligne, la colonne, et le message complet (« Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent »). Et la dernière ligne — `0 Erreur(s)` — rappelle le choix de sévérité : le build réussit, le diagnostic est un garde-fou, pas un marteau.

**Le point décisif** : cette sortie est produite par `dotnet build`, la commande la plus ordinaire du monde .NET. Aucune mention d'outil externe, aucune étape de lint. Le diagnostic est arrivé **avec la compilation elle-même** — c'est ce que Python ne peut pas offrir par construction, et que la section C détaille.

## B1 — Le canal API : le `Verifier` compile un source et rend son verdict

`dotnet build` est le canal *intégration*. Il en existe un second : l'**API Roslyn** elle-même — c'est ce qu'utilisent l'IDE (le soulignement jaune pendant la frappe) et les tests. Le projet `AgentGuard.Verifier` (committé) prend deux fichiers sources en argument, les compile en mémoire via `CSharpCompilation` + `WithAnalyzers`, et rend un verdict par fichier.

On lui soumet les deux variantes du terrain : le fichier fautif du Demo, et la version corrigée (`samples/AgentWorkerCorrige.cs`, committée — le même worker, mais `async`/`await` de bout en bout).

In [6]:
var verdicts = Shell.Run(here, "dotnet",
    "run --project AgentGuard.Verifier -- AgentGuard.Demo/Program.cs AgentGuard.Verifier/samples/AgentWorkerCorrige.cs");
Console.WriteLine(verdicts);

[Program.cs] VERDICT : 2 diagnostic(s) AGENTGUARD001
    AGENTGUARD001 @ 24:22  Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent
    AGENTGUARD001 @ 25:16  Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent
[AgentWorkerCorrige.cs] VERDICT : PROPRE -- aucun blocage synchrone detecte



### Lecture des verdicts

Deux lignes de verdict, chacune adossée à son fichier :

- `Program.cs` (le terrain fautif) : **2 diagnostics AGENTGUARD001**, aux positions 24:22 et 25:16 — les mêmes que le canal build ;
- `AgentWorkerCorrige.cs` (la version corrigée) : **PROPRE**, aucun blocage détecté.

**Deux canaux, un seul moteur** : l'analyseur est le même code de ~40 lignes ; ce qui change, c'est qui l'héberge — MSBuild pour le canal build, l'API pour le canal tests/IDE. Un test d'intégration continue peut faire exactement ce que fait ce Verifier : compiler les sources générés par l'agent et exiger zéro AGENTGUARD001. Le garde-fou devient alors **exécutable en CI sur du code qui n'existe même pas sous forme de projet** — du code généré à la volée, collé dans un `CSharpCompilation`, verdict immédiat.

In [7]:
var corrige = File.ReadAllText(Path.Combine(here, "AgentGuard.Verifier", "samples", "AgentWorkerCorrige.cs"));
Console.WriteLine(corrige);

// Version corrigee du terrain fautif (AgentGuard.Demo/Program.cs) :
// le worker attend la tache au lieu de la bloquer. C'est la version
// attendue "propre" au verdict du Verifier.

using System;
using System.Threading;
using System.Threading.Channels;
using System.Threading.Tasks;

public static class AgentWorkerCorrige
{
    // Le fix : async/await de bout en bout. Le thread rend la main pendant
    // l'attente au lieu de se bloquer -- plus de deadlock possible, et le
    // pipeline reste fluide sous charge.
    public static async Task<string> TranslateAsync(
        ChannelReader<string> inbound, CancellationToken ct = default)
    {
        var prompt = await inbound.ReadAsync(ct);
        return await CallLlmAsync(prompt, ct);
    }

    private static async Task<string> CallLlmAsync(string prompt, CancellationToken ct)
    {
        await Task.Delay(50, ct);       // simule la latence de l'appel LLM
        return $"[LLM] {prompt}";
    }
}



### Le fix : `await`, et pourquoi il suffit

La version corrigée ne change pas la logique — seulement le contrat :

- `TranslateSync` devient `TranslateAsync` : `async Task<string>` au lieu de `string` ;
- chaque `.Result` devient un `await` : `await inbound.ReadAsync(ct)` et `await CallLlmAsync(prompt, ct)` ;
- un `CancellationToken` voyage de bout en bout — bonus de cohérence avec la série.

**Pourquoi le deadlock disparaît** : `await` ne bloque pas le thread — il **rend la main**. Le thread retourne au pool, la continuation s'exécute quand la tâche finit, sur un thread disponible. Le pipeline reste fluide sous charge, et le verdict PROPRE du Verifier confirme que l'analyseur reconnaît la correction.

Quant au *code fixer* d'IDE (l'ampoule qui transformerait `.Result` en `await` d'un clic) : il vit côté IDE, invisible de `dotnet build` — le corriger automatiquement exige de réécrire la méthode englobante en `async`, une transformation trop contextuelle pour être montrée proprement ici. Le fix manuel ci-dessus est celui que l'ampoule elle-même proposerait.

### Exercice 2 — Prouver l'exemption : un `Outcome<T>` monadique ne doit PAS être signalé

L'étage sémantique de l'analyseur (`MetadataName is "Task" or "Task`1"`) existe pour éviter les faux positifs : un type fonctionnel `Outcome<T>` qui expose une propriété `Result` est **légal et non bloquant** — l'analyseur ne doit pas le signaler.

**Objectif** : le prouver par l'exécution, puis expliquer la ligne qui fait l'exemption.

**Indices** :
- `# Indice` : la cellule ci-dessous définit un `Outcome<T>` minimal avec une propriété `Result` — elle compile et s'exécute sans avertissement ; pourquoi le build du Demo n'a-t-il rien dit sur elle ?
- `# Etape 1` : écrire un source de test `samples/exercice2.cs` utilisant `Outcome<string>.Result`, et le passer au Verifier — verdict attendu : PROPRE.
- `# Etape 2` : dans `TaskResultBlockAnalyzer.AnalyzeNode`, identifier la ligne exacte qui évite le faux positif, et formuler en une phrase ce qu'elle vérifie.
- `# Etape 3` (pour aller plus loin) : supprimer mentalement cette ligne — quels nouveaux diagnostics apparaîtraient dans le corpus de la série ?

In [8]:
// Exercice 2 -- terrain : un Outcome<T> monadique avec propriete .Result.
// Ce type N'EST PAS une Task : bloquer n'a pas de sens ici, et l'analyseur
// ne le signale pas. TODO etudiant : verifier ce verdict via le Verifier.
public readonly record struct Outcome<T>(bool IsOk, T Value)
{
    public T Result => Value;   // propriete .Result SANS Task -- legal
}

var o = new Outcome<string>(true, "aucun blocage ici");
Console.WriteLine($"Outcome<string>.Result = {o.Result}");
Console.WriteLine("Exercice a completer : passer ce type au Verifier et nommer la ligne de l'exemption");

Outcome<string>.Result = aucun blocage ici


Exercice a completer : passer ce type au Verifier et nommer la ligne de l'exemption


## C — Le contraste Python : le même défaut, deux écosystèmes

La classe de défaut est universelle : **bloquer une tâche asynchrone depuis du code synchrone**. En Python, une coroutine qui appelle une fonction bloquante (`time.sleep`, `requests.get`) produit exactement le même gel de l'événementiel. Voyons ce que chaque écosystème propose.

In [9]:
// Verifie sur cette machine : ruff est-il disponible sans installation ?
var probe = Shell.Run(here, "where", "ruff");
var ruffPresent = probe.Contains("ruff", StringComparison.OrdinalIgnoreCase);
Console.WriteLine(probe);
Console.WriteLine(ruffPresent
    ? "-> ruff present sur cette machine"
    : "-> ruff ABSENT : en Python le garde-fou est un outil a installer soi-meme");

Information : impossible de trouver des fichiers pour le(s) modèle(s) spécifié(s).



-> ruff ABSENT : en Python le garde-fou est un outil a installer soi-meme


### La comparaison, vérifiée

Sur cette machine, `ruff` n'est pas installé — ce qui est déjà la moitié de la thèse : en Python, le garde-fou est un **outil à adopter** (pip install, config, pipeline). La règle qui couvrirait notre défaut existe et est documentée : **`ASYNC101`** (famille `flake8-async`, intégrée à ruff) — « les fonctions async ne doivent pas appeler de méthodes synchrones bloquantes ». Vérifié contre la documentation ruff : cette règle fait partie des règles `ASYNC`, **non activées par défaut** — il faut les sélectionner explicitement (`[tool.ruff.lint] select = ["ASYNC"]`).

| Aspect | Python (ruff `ASYNC101` / mypy) | .NET (Roslyn `AGENTGUARD001`) |
|---|---|---|
| Où vit le garde-fou | **Hors** compilation — outil distinct | **Dans** la compilation |
| Adoption | installer + configurer + ajouter au pipeline | une `ProjectReference` |
| Activation | opt-in par règle (`select = ["ASYNC"]`) | `isEnabledByDefault: true` |
| Moment du verdict | quand on lance le lint | à chaque `dotnet build` |
| Compréhension des types | partielle (analyse statique du source) | **totale** (modèle sémantique du compilateur) |
| Faux positifs `Result<T>` | possibles (analyse sans compilateur) | évités (l'analyseur demande au compilateur) |

La nuance honnête : l'écosystème Python compense par la **richesse des règles prêtes à l'emploi** (des centaines, maintenues par la communauté) là où notre analyseur est une règle maison de 40 lignes. Le déplacement profond n'est pas la quantité mais **le moment** : en .NET, le diagnostic est *constitutif* de la chaîne qui produit le binaire — impossible de compiler sans passer devant le garde-fou.

## D — Le réflexe pour du code généré par agent

Résumé du geste complet, tel qu'un pipeline d'agent .NET peut l'adopter :

1. **Un analyseur par classe de défaut** — ici le blocage de Task ; demain l'appel HTTP sans timeout (leçon du notebook 01), le secret en dur (leçon de la série sécurité), la désérialisation non bornée. Quarante lignes chacune.
2. **Référencé dans le projet** — `OutputItemType="Analyzer"` : chaque `dotnet build`, de chaque machine, rend le verdict. L'agent qui génère du code reçoit le diagnostic **dans la sortie de son propre build** — boucle de correction immédiate, sans review humain dans la boucle.
3. **Exécutable en CI sur du code volatile** — le canal API (Verifier) permet de filtrer le code généré *avant même* qu'il n'atteigne un projet : génération → compilation en mémoire → verdict → garder ou régénérer.

C'est la réponse .NET à la question du garde-fou de code généré : ne pas ajouter un outil à côté du compilateur — **mettre la règle dans le compilateur**.

### Exercice 3 — `AGENTGUARD002` : `async void` hors gestionnaire d'événement

Second pattern d'agent classique : `async void FaitUneChose()` — la méthode ressemble à une `async Task`, mais ses exceptions **échappent à tout mécanisme d'attente** : non observables, elles font planter le process ; la tâche n'est pas attendable, donc non testable. Légitime seulement pour les gestionnaires d'événements (`async void Button_Click`).

**Objectif** : écrire le squelette du second analyseur.

**Indices** :
- `# Indice` : s'abonner à `SyntaxKind.MethodDeclaration` ; vérifier le modificateur `async`, le type de retour `void`, et le nom du premier paramètre (un handler d'événement reçoit `object sender, EventArgs e` — exempter ce cas).
- `# Etape 1` : lister à la main les formes `async void` du dépôt qui seraient signalées (grep `async void`).
- `# Etape 2` : écrire le squelette ci-dessous en code réel dans une copie de l'analyseur, avec son `DiagnosticDescriptor` (`AGENTGUARD002`).
- `# Etape 3` : brancher le résultat dans le projet Demo et vérifier le verdict sur un source contenant `async void Travail() { ... }`.

In [10]:
// Exercice 3 a completer -- squelette d'AGENTGUARD002.
//
// private static readonly DiagnosticDescriptor Rule2 = new(
//     "AGENTGUARD002",
//     "async void hors gestionnaire d'evenement",
//     "La methode async void '{0}' echappe a toute attente -- exceptions non observees",
//     "Agentisme", DiagnosticSeverity.Warning, isEnabledByDefault: true);
//
// public override void Initialize(AnalysisContext context)
//     => context.RegisterSyntaxNodeAction(AnalyzeMethod, SyntaxKind.MethodDeclaration);
//
// private static void AnalyzeMethod(SyntaxNodeAnalysisContext ctx)
// {
//     var m = (MethodDeclarationSyntax)ctx.Node;
//     // TODO etudiant : modificateur async present ?
//     // TODO etudiant : type de retour "void" ?
//     // TODO etudiant : parametres (object, EventArgs) => exemption handler
//     // TODO etudiant : ctx.ReportDiagnostic(... nom de la methode ...)
// }
Console.WriteLine("Exercice a completer : AGENTGUARD002 async void hors handler");

Exercice a completer : AGENTGUARD002 async void hors handler


## Conclusion

Ce notebook a démontré, exécution à l'appui :

- un **analyseur Roslyn réel** de ~40 lignes (`TaskResultBlockAnalyzer`) attrape le pattern de deadlock le plus récurrent du code d'agent généré ;
- sur le **canal build**, `dotnet build` rend deux `AGENTGUARD001` sans aucun outil supplémentaire — la thèse DANS-la-compilation ;
- sur le **canal API**, le `Verifier` compile faulty/corrigé et rend ses verdicts (2 diagnostics / PROPRE) — le même moteur que l'IDE et que vos tests CI ;
- le **fix** est minimal : `async`/`await` de bout en bout, et l'analyseur reconnaît la correction ;
- le **contraste Python** est un déplacement de moment : là où ruff exige une adoption et une activation par règle, l'analyseur .NET est constitutif de la chaîne de build.

Trois exercices restent ouverts — la variante furtive `GetAwaiter().GetResult()`, l'exemption `Result<T>`, et l'analyseur `async void` — chacun étendant le même squelette à deux étages (syntaxique puis sémantique).

**Pour aller plus loin** : la série Aspire (notebooks 01-05) fournit le terrain d'agent ; l'Epic #10473 tient la parité Python/C# des garde-fous ; et la documentation Roslyn `DiagnosticAnalyzer` couvre les code fixers et les tests (`Microsoft.CodeAnalysis.Analyzer.Testing`).

See #10473